# NLP and Text with Deep Learning
Natural Language Processing with deep learning spans text classification, sequence labeling, machine translation, question answering, and language generation. This notebook covers key architectures and techniques: embeddings, LSTM-based NLP, BERT, GPT, and sequence-to-sequence models.

## 1. Word Embeddings
Dense vector representations of words that capture semantic meaning.
- **Word2Vec**: Skip-gram and CBOW — predict surrounding words from center word (or vice versa)
- **GloVe**: Global Vectors — factorizes the global word co-occurrence matrix
- **FastText**: Character n-gram embeddings — handles morphology and out-of-vocabulary words
- **Contextual Embeddings (ELMo, BERT)**: The embedding of a word depends on its surrounding context — "bank" near "river" vs "bank" near "money" gets different vectors

## 2. NLP Task Taxonomy
| Task | Output | Model |
|---|---|---|
| Text Classification | Label | BERT fine-tuned |
| NER | BIO tags per token | BERT + CRF |
| Machine Translation | Target sentence | Transformer |
| Question Answering | Span of text | BERT extractive QA |
| Text Summarization | Summary text | BART, T5 |
| Language Generation | Next tokens | GPT |

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

vocab_size = 10000
embed_dim = 128
max_len = 200

# Text Classification using LSTM
model = models.Sequential([
    layers.Embedding(vocab_size, embed_dim, input_length=max_len),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')   # Binary classification
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 3. BERT for NLP Tasks
Fine-tuning BERT follows a unified pattern:
1. Tokenize input with [CLS] and [SEP] tokens
2. Pass through BERT encoder
3. Use [CLS] embedding for classification, token embeddings for token-level tasks

**Text Classification**: [CLS] → Dense → Softmax
**NER/POS Tagging**: Each token embedding → Dense → Softmax (per token)
**Extractive QA (SQuAD)**: Each token gets start/end logits; answer is the highest-scoring span

## 4. Sequence-to-Sequence (Seq2Seq) Models
Used for machine translation, summarization, dialogue.
- **Encoder**: encodes source sequence to context vector
- **Decoder**: generates target sequence autoregressively
- **Attention**: allows decoder to attend selectively to different encoder positions at each generation step (Bahdanau/Luong attention)
- Modern Seq2Seq uses the full Transformer Encoder-Decoder (as in BART, T5, mBART)

In [ ]:
# Using HuggingFace pipeline (conceptual — requires transformers library)
# pip install transformers

from transformers import pipeline

# Sentiment classification (BERT fine-tuned)
# clf = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")
# result = clf("I love deep learning!")
# print(result)  # [{'label': 'POSITIVE', 'score': 0.9998}]

# Named Entity Recognition
# ner = pipeline("ner", model="dslim/bert-base-NER")
# ner("Apple Inc was founded by Steve Jobs in Cupertino.")

# Summarization (BART)
# summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
# summarizer("Long document text here...")

print("HuggingFace pipelines ready (uncomment after installing transformers)")

# Conclusions and Key Takeaways
- Contextual embeddings (BERT, RoBERTa) have replaced static embeddings for nearly all NLP tasks.
- The pre-train + fine-tune paradigm dominates: a single pre-trained model adapts to dozens of tasks.
- Seq2Seq with Transformer Encoder-Decoder is the gold standard for generation tasks.
- Large Language Models (GPT-3/4, LLaMA, Gemini) show emergent capabilities through massive scale.

# Pros and Cons
**Pros:**
- Transfer learning allows high performance with small labeled datasets
- Transformers parallelize well, enabling training on terabytes of text
- Single pre-trained model adapts to many tasks (classification, generation, QA, translation)

**Cons:**
- Large models require expensive GPU infrastructure for both training and inference
- Prone to hallucination — generating confident but factually incorrect text
- Biases from pre-training data can be inherited and amplified

# 15 Interview Questions and Answers

1. **What is the difference between Word2Vec Skip-gram and CBOW?**
   *Answer*: Skip-gram predicts surrounding context words given the center word (better for rare words). CBOW predicts the center word from the surrounding context (faster, better for frequent words).

2. **What is an out-of-vocabulary (OOV) problem and how does FastText address it?**
   *Answer*: Standard embeddings cannot represent words not seen during training. FastText decomposes words into character n-grams and sums their embeddings, allowing it to represent any word including misspellings and morphological variants.

3. **What does [CLS] represent in BERT?**
   *Answer*: A special classification token prepended to every input sequence. BERT's final hidden state for [CLS] aggregates information from the full sequence and is used as the representation for sentence-level tasks.

4. **How does BERT represent each input token?**
   *Answer*: Each token's input representation = Token Embedding + Segment Embedding (sentence A or B) + Positional Embedding.

5. **What is Named Entity Recognition (NER)?**
   *Answer*: A sequence labeling task that identifies and classifies named entities (Person, Organization, Location, Date) in text. BERT fine-tuned for NER outputs a label per token using BIO tagging.

6. **What is Extractive vs Abstractive Summarization?**
   *Answer*: Extractive selects and concatenates important sentences verbatim from the source. Abstractive generates a new summary in novel words, often more concise but requiring strong generation capabilities (e.g., BART).

7. **What is Beam Search?**
   *Answer*: A decoding algorithm that maintains the top-k (beam width) most probable partial sequences at each generation step, rather than greedily selecting only the single best token.

8. **What is the difference between BERT and RoBERTa?**
   *Answer*: RoBERTa (Robustly Optimized BERT) is BERT trained with: larger batch sizes, longer training, dynamic masking, no NSP task, and much more data. This yields improved performance across NLP benchmarks.

9. **What is RLHF (Reinforcement Learning from Human Feedback)?**
   *Answer*: A technique to align language model outputs with human preferences. A reward model is trained on human preference rankings, then used to fine-tune the LM via PPO reinforcement learning. Used in ChatGPT, Claude etc.

10. **What is the BPE (Byte Pair Encoding) tokenizer?**
    *Answer*: A subword tokenization algorithm. Starts with characters, then iteratively merges the most frequent adjacent pairs. Balances vocabulary size with OOV coverage, used by GPT, BART, and most modern LLMs.

11. **What is the difference between encoder-only (BERT), decoder-only (GPT), and encoder-decoder (T5)?**
    *Answer*: Encoder-only sees full bidirectional context — ideal for understanding tasks. Decoder-only generates left-to-right — ideal for generation. Encoder-decoder provides the richest cross-attention for seq2seq tasks like translation/summarization.

12. **What is temperature in language model generation?**
    *Answer*: Divides logits before softmax. T<1: sharper distribution, more confident/deterministic outputs. T>1: flatter distribution, more diverse/creative (but potentially less coherent) outputs.

13. **What is Perplexity as a language model metric?**
    *Answer*: The exponent of the average negative log-likelihood per token: exp(-1/N * sum log P(token_i)). Lower perplexity means the model assigns higher probability to the test text — i.e., it is less "surprised."

14. **What is RAG (Retrieval Augmented Generation)?**
    *Answer*: Augments the LLM generation process with a retrieval step. Given a query, relevant documents are fetched from a vector database and concatenated with the query as context before generation — reducing hallucination.

15. **What is the "needle in a haystack" problem for LLMs?**
    *Answer*: LLMs often fail to retrieve a specific piece of information placed in the middle of a very long context window, even if it fits in the context. Performance degrades significantly for positions far from the start and end of the context.
